# 10 - Business Recommendations

Business interpretation and retention recommendations from the outputs of notebooks 07-09.

## SECTION 01 - Configuration and Reproducibility

In [14]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

RANDOM_STATE = 42
TARGET = 'Churned'

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_FILE = PROJECT_ROOT / 'data' / 'processed' / 'customer_churn_feature_engineered.csv'
SEGMENTS_FILE = PROJECT_ROOT / 'models' / 'segmentation' / 'customer_segments.csv'
PROFILE_FILE = PROJECT_ROOT / 'models' / 'segmentation' / 'segment_profile.csv'
FEATURE_SUMMARY_FILE = PROJECT_ROOT / 'models' / 'segmentation' / 'segment_feature_summary.csv'
SEGMENT_SHAP_FILE = PROJECT_ROOT / 'models' / 'segmentation' / 'segment_shap_summary.csv'
METADATA_FILE = PROJECT_ROOT / 'models' / 'segmentation' / 'segmentation_metadata.json'
SHAP_GLOBAL_FILE = PROJECT_ROOT / 'models' / 'shap' / 'shap_global_summary.csv'
SHAP_IMPORTANCE_FILE = PROJECT_ROOT / 'models' / 'shap' / 'shap_feature_importance.csv'
SHAP_LOCAL_FILE = PROJECT_ROOT / 'models' / 'shap' / 'shap_local_explanations.csv'

BUSINESS_DIR = PROJECT_ROOT / 'reports' / 'business'
BUSINESS_FIGURES_DIR = PROJECT_ROOT / 'reports' / 'figures' / 'business'
BUSINESS_DIR.mkdir(parents=True, exist_ok=True)
BUSINESS_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

if RANDOM_STATE != 42:
    raise ValueError('RANDOM_STATE must be 42.')
if TARGET != 'Churned':
    raise ValueError('TARGET must be Churned.')

print(f'Project root: {PROJECT_ROOT}')
print(f'Business output directory: {BUSINESS_DIR}')

Project root: d:\Desktop\Projects\customer-churn-analytics
Business output directory: d:\Desktop\Projects\customer-churn-analytics\reports\business


## SECTION 02 - Load and Validate Business Inputs

In [15]:
required_input_files = [
    DATA_FILE,
    SEGMENTS_FILE,
    PROFILE_FILE,
    FEATURE_SUMMARY_FILE,
    METADATA_FILE,
]

missing_input_files = [
    path for path in required_input_files
    if not path.exists()
]

if missing_input_files:
    raise FileNotFoundError(
        f'Missing required business inputs: {missing_input_files}'
    )

customer_data = pd.read_csv(DATA_FILE)
customer_segments = pd.read_csv(SEGMENTS_FILE)
segment_profile = pd.read_csv(PROFILE_FILE)
segment_feature_summary = pd.read_csv(FEATURE_SUMMARY_FILE)

with METADATA_FILE.open(encoding='utf-8') as metadata_file:
    segmentation_metadata = json.load(metadata_file)

segment_shap_summary = (
    pd.read_csv(SEGMENT_SHAP_FILE)
    if SEGMENT_SHAP_FILE.exists()
    else None
)

shap_global_summary = (
    pd.read_csv(SHAP_GLOBAL_FILE)
    if SHAP_GLOBAL_FILE.exists()
    else None
)

shap_feature_importance = (
    pd.read_csv(SHAP_IMPORTANCE_FILE)
    if SHAP_IMPORTANCE_FILE.exists()
    else None
)

if segment_profile['Customer_Count'].sum() != len(customer_data):
    raise ValueError('Segment customer counts do not sum to the dataset row count.')
if segment_profile['Churned_Count'].sum() != int(customer_data[TARGET].sum()):
    raise ValueError('Segment churned counts do not sum to the total number of churned customers.')
if (segment_profile['Customer_Count'] <= 0).any():
    raise ValueError('Segment customer counts must be greater than zero.')
calculated_segment_churn = segment_profile['Churned_Count'] / segment_profile['Customer_Count']
if not np.allclose(segment_profile['Churn_Rate'], calculated_segment_churn, atol=1e-6):
    raise ValueError('Segment churn rates do not match churned/customer counts.')
if not np.isclose(segment_profile['Customer_Percentage'].sum(), 100.0, atol=0.1):
    raise ValueError('Segment customer percentages do not sum to approximately 100%.')
if not segment_profile['Churn_Rate'].between(0, 1).all():
    raise ValueError('Segment churn rates must be between 0 and 1.')
if set(customer_segments['Segment'].unique()) != set(segment_profile['Segment'].unique()):
    raise ValueError('Segment IDs do not match between customer segmentation and segment profile.')
if 'Segment' not in segment_feature_summary.columns:
    raise ValueError('Segment feature summary must contain Segment.')

required_metadata = {'algorithm', 'selected_k', 'features', 'random_state', 'silhouette_score', 'dataset_rows'}
if not required_metadata.issubset(segmentation_metadata):
    raise ValueError(f'Segmentation metadata is missing keys: {sorted(required_metadata - set(segmentation_metadata))}.')
if segmentation_metadata['algorithm'] != 'KMeans':
    raise ValueError('The segmentation algorithm must be KMeans.')
if segmentation_metadata['dataset_rows'] != len(customer_data):
    raise ValueError('Segmentation metadata dataset_rows does not match the dataset length.')
if segmentation_metadata['selected_k'] != segment_profile['Segment'].nunique():
    raise ValueError('Segmentation metadata selected_k does not match the number of segments.')
if segmentation_metadata['random_state'] != RANDOM_STATE:
    raise ValueError('Segmentation metadata random_state does not match the project random state.')

rank_series = segment_profile['Retention_Priority_Rank']
if rank_series.isnull().any():
    raise ValueError('Retention priority ranks contain null values.')
rank_values = pd.to_numeric(rank_series, errors='coerce')
if rank_values.isna().any():
    raise ValueError('Retention priority ranks must be numeric and non-null.')
if not np.all(np.isclose(rank_values, np.round(rank_values))):
    raise ValueError('Retention priority ranks must be integers.')
rank_values = rank_values.astype(int)
expected_ranks = set(range(1, len(segment_profile) + 1))
actual_ranks = set(rank_values.unique())
if len(actual_ranks) != len(rank_values) or actual_ranks != expected_ranks:
    raise ValueError('Retention priority ranks are not unique and consecutive.')

print(
    f"Loaded {len(customer_data):,} customers and {segment_profile['Segment'].nunique()} segments."
)


Loaded 50,000 customers and 2 segments.


## SECTION 03 - Overall Churn Business Overview

In [16]:
total_customers = int(len(customer_data))
churned_customers = int(customer_data[TARGET].sum())
active_customers = total_customers - churned_customers
overall_churn_rate = churned_customers / total_customers

overall_churn_summary = pd.DataFrame([
    {'Metric': 'Total Customers', 'Value': total_customers},
    {'Metric': 'Active Customers', 'Value': active_customers},
    {'Metric': 'Churned Customers', 'Value': churned_customers},
    {'Metric': 'Overall Churn Rate', 'Value': overall_churn_rate},
    {'Metric': 'Average Lifetime Value', 'Value': customer_data['Lifetime_Value'].mean()},
    {'Metric': 'Median Lifetime Value', 'Value': customer_data['Lifetime_Value'].median()},
    {'Metric': 'Average Engagement Score', 'Value': customer_data['Engagement_Score'].mean()},
])

print(f'Overall churn rate: {overall_churn_rate:.2%}')
print(f'The customer base contains {churned_customers:,} churned customers out of {total_customers:,} total customers.')
overall_churn_summary

Overall churn rate: 28.90%
The customer base contains 14,450 churned customers out of 50,000 total customers.


,Metric,Value
0,Total Customers,50000.000000
1,Active Customers,35550.000000
2,Churned Customers,14450.000000
3,Overall Churn Rate,0.289000
4,Average Lifetime Value,1440.626292
5,Median Lifetime Value,1243.415000
6,Average Engagement Score,0.300740


## SECTION 04 - Segment Business Performance

In [17]:
segment_business_summary = segment_profile[[
    'Segment', 'Segment_Name', 'Customer_Count', 'Customer_Percentage',
    'Churned_Count', 'Churn_Rate', 'Avg_Lifetime_Value',
    'Avg_Engagement_Score', 'Business_Priority', 'Retention_Priority_Score',
    'Retention_Priority_Rank', 'Recommended_Action',
]].copy()
segment_business_summary['Churn_Rate_vs_Overall'] = segment_business_summary['Churn_Rate'] - overall_churn_rate
segment_business_summary['Churn_Classification'] = np.select(
    [
        segment_business_summary['Churn_Rate'] > overall_churn_rate,
        segment_business_summary['Churn_Rate'] < overall_churn_rate,
    ],
    ['Above Overall Churn', 'Below Overall Churn'],
    default='At Overall Churn',
)
segment_business_summary = segment_business_summary[[
    'Segment', 'Segment_Name', 'Customer_Count', 'Customer_Percentage',
    'Churned_Count', 'Churn_Rate', 'Churn_Rate_vs_Overall',
    'Avg_Lifetime_Value', 'Avg_Engagement_Score', 'Business_Priority',
    'Retention_Priority_Score', 'Retention_Priority_Rank',
    'Recommended_Action', 'Churn_Classification',
]]
segment_business_summary

,Segment,Segment_Name,Customer_Count,Customer_Percentage,Churned_Count,Churn_Rate,Churn_Rate_vs_Overall,Avg_Lifetime_Value,Avg_Engagement_Score,Business_Priority,Retention_Priority_Score,Retention_Priority_Rank,Recommended_Action,Churn_Classification
0,0,Loyal High-Value Customers (Segment 0),18506,37.012,3787,0.204636,-0.084364,2102.157397,0.440655,MAINTAIN,7.960870e+06,2,Protect loyalty with relevant benefits and mai...,Below Overall Churn
1,1,At-Risk Customers (Segment 1),31494,62.988,10663,0.338572,0.049572,1051.907976,0.218525,HIGH,1.121649e+07,1,Investigate friction and test targeted retenti...,Above Overall Churn


## SECTION 05 - Churn Drivers from SHAP

SHAP describes contributions to model predictions; it does not establish causal relationships.

In [18]:
churn_driver_source = None
if shap_global_summary is not None and {'Feature', 'Mean_Abs_SHAP'}.issubset(shap_global_summary.columns):
    churn_driver_source = shap_global_summary[['Feature', 'Mean_Abs_SHAP']].copy()
elif shap_feature_importance is not None and {'Feature', 'Mean_Abs_SHAP'}.issubset(shap_feature_importance.columns):
    churn_driver_source = shap_feature_importance[['Feature', 'Mean_Abs_SHAP']].copy()
elif segment_shap_summary is not None:
    churn_driver_source = segment_shap_summary.groupby('Feature', as_index=False)['Mean_Abs_SHAP'].mean()

if churn_driver_source is None:
    churn_driver_summary = pd.DataFrame(columns=['Rank', 'Feature', 'Mean_Abs_SHAP'])
    print('SHAP driver summary unavailable.')
else:
    churn_driver_summary = churn_driver_source.dropna(subset=['Feature', 'Mean_Abs_SHAP']).copy()
    churn_driver_summary['Mean_Abs_SHAP'] = pd.to_numeric(churn_driver_summary['Mean_Abs_SHAP'], errors='raise')
    churn_driver_summary = churn_driver_summary.sort_values('Mean_Abs_SHAP', ascending=False).head(5)
    churn_driver_summary.insert(0, 'Rank', range(1, len(churn_driver_summary) + 1))
    churn_driver_summary = churn_driver_summary[['Rank', 'Feature', 'Mean_Abs_SHAP']].reset_index(drop=True)

churn_driver_summary

,Rank,Feature,Mean_Abs_SHAP
0,1,num__Lifetime_Value,0.835753
1,2,num__Customer_Service_Calls,0.806775
2,3,num__Cart_Abandonment_Rate,0.440911
3,4,num__Age,0.416268
4,5,num__Cart_Recency_Interaction,0.327927


## SECTION 06 - Segment x Churn x SHAP Analysis

In [19]:
behavior_columns = [
    'Login_Frequency_mean',
    'Days_Since_Last_Purchase_mean',
    'Cart_Abandonment_Rate_mean',
]
behavior_columns = [column for column in behavior_columns if column in segment_feature_summary.columns]
segment_churn_driver_records = []

if segment_shap_summary is not None:
    feature_summary_by_segment = segment_feature_summary.set_index('Segment')
    for _, profile_row in segment_profile.iterrows():
        segment_id = profile_row['Segment']
        segment_drivers = segment_shap_summary.loc[
            segment_shap_summary['Segment'] == segment_id
        ].sort_values('Mean_Abs_SHAP', ascending=False).head(3)
        feature_list = segment_drivers['Feature'].astype(str).tolist()
        main_signals = '; '.join(feature_list)
        behavior_values = []
        if segment_id in feature_summary_by_segment.index:
            behavior_row = feature_summary_by_segment.loc[segment_id]
            for column in behavior_columns:
                label = column.removesuffix('_mean').replace('_', ' ')
                behavior_values.append(f'{label}: {behavior_row[column]:.2f}')
        behavior_profile = '; '.join(behavior_values)
        if feature_list:
            interpretation = (
                f'Segment {segment_id} has a churn rate of '
                f'{profile_row["Churn_Rate"]:.1%}, '
                f'an average lifetime value of '
                f'{profile_row["Avg_Lifetime_Value"]:,.2f}, '
                f'and an average engagement score of '
                f'{profile_row["Avg_Engagement_Score"]:.2f}. '
                f'Its behavioral profile ({behavior_profile}) can be reviewed '
                f'alongside the model signals ({main_signals}) to identify '
                'potential areas for further business investigation. '
                'SHAP describes model prediction contributions and does not '
                'establish causal relationships.'
)
        else:
            interpretation = (
                f'Segment {segment_id} has a churn rate of {profile_row["Churn_Rate"]:.1%} and '
                f'average lifetime value of {profile_row["Avg_Lifetime_Value"]:,.2f}; '
                'no segment-level SHAP drivers were available for a deeper model interpretation.'
            )
        segment_churn_driver_records.append({
            'Segment': segment_id,
            'Segment_Name': profile_row['Segment_Name'],
            'Churn_Rate': profile_row['Churn_Rate'],
            'Avg_Lifetime_Value': profile_row['Avg_Lifetime_Value'],
            'Avg_Engagement_Score': profile_row['Avg_Engagement_Score'],
            'Behavior_Profile': behavior_profile,
            'Main_Churn_Signals': main_signals,
            'Business_Interpretation': interpretation,
        })
else:
    print('Continue without segment-level SHAP analysis.')

segment_churn_driver_summary = pd.DataFrame(
    segment_churn_driver_records,
    columns=[
        'Segment', 'Segment_Name', 'Churn_Rate', 'Avg_Lifetime_Value',
        'Avg_Engagement_Score', 'Behavior_Profile', 'Main_Churn_Signals',
        'Business_Interpretation',
    ],
)

if segment_shap_summary is not None:
    required_shap_columns = {
        'Segment', 'Feature', 'Mean_Abs_SHAP'
    }

    missing_columns = (
        required_shap_columns - set(segment_shap_summary.columns)
    )

    if missing_columns:
        raise ValueError(
            'Segment SHAP summary is missing columns: '
            f'{sorted(missing_columns)}'
        )
    
segment_churn_driver_summary


,Segment,Segment_Name,Churn_Rate,Avg_Lifetime_Value,Avg_Engagement_Score,Behavior_Profile,Main_Churn_Signals,Business_Interpretation
0,0,Loyal High-Value Customers (Segment 0),0.204636,2102.157397,0.440655,Login Frequency: 18.47; Days Since Last Purcha...,num__Customer_Service_Calls; num__Lifetime_Val...,"Segment 0 has a churn rate of 20.5%, an averag..."
1,1,At-Risk Customers (Segment 1),0.338572,1051.907976,0.218525,Login Frequency: 7.60; Days Since Last Purchas...,,Segment 1 has a churn rate of 33.9% and averag...


## SECTION 07 - Retention Priority

`Retention_Priority_Score` is a heuristic business score, not a churn probability or an ML prediction score.

In [20]:
segment_retention_priority = segment_profile[[
    'Segment', 'Segment_Name', 'Customer_Count', 'Customer_Percentage',
    'Churn_Rate', 'Avg_Lifetime_Value', 'Avg_Engagement_Score',
    'Business_Priority', 'Retention_Priority_Score',
    'Retention_Priority_Rank', 'Recommended_Action',
]].copy()
segment_retention_priority = segment_retention_priority.sort_values(
    'Retention_Priority_Rank', ascending=True
).reset_index(drop=True)
segment_retention_priority

,Segment,Segment_Name,Customer_Count,Customer_Percentage,Churn_Rate,Avg_Lifetime_Value,Avg_Engagement_Score,Business_Priority,Retention_Priority_Score,Retention_Priority_Rank,Recommended_Action
0,1,At-Risk Customers (Segment 1),31494,62.988,0.338572,1051.907976,0.218525,HIGH,1.121649e+07,1,Investigate friction and test targeted retenti...
1,0,Loyal High-Value Customers (Segment 0),18506,37.012,0.204636,2102.157397,0.440655,MAINTAIN,7.960870e+06,2,Protect loyalty with relevant benefits and mai...


## SECTION 08 - Business Recommendations

In [21]:
segment_signals = {}
if not segment_churn_driver_summary.empty:
    segment_signals = segment_churn_driver_summary.set_index('Segment')['Main_Churn_Signals'].to_dict()
overall_driver_signal = (
    str(churn_driver_summary.iloc[0]['Feature'])
    if not churn_driver_summary.empty
    else 'Not available'
)

def _signal_focus(main_signal: str) -> str:
    normalized = str(main_signal).lower()
    if 'days_since_last_purchase' in normalized or 'recency' in normalized:
        return 're-engagement and purchase recency'
    if 'cart_abandonment' in normalized or 'abandonment' in normalized:
        return 'checkout friction and cart recovery'
    if 'customer_service_calls' in normalized or 'service' in normalized:
        return 'service recovery and support escalation'
    if 'lifetime_value' in normalized or 'value' in normalized:
        return 'high-value retention and loyalty protection'
    return 'early warning and retention triggers'

recommendation_records = []
for _, row in segment_profile.iterrows():
    churn_rate = row['Churn_Rate']
    churn_gap = churn_rate - overall_churn_rate
    value_is_high = row['Avg_Lifetime_Value'] >= segment_profile['Avg_Lifetime_Value'].median()
    segment_signal = str(segment_signals.get(row['Segment'], '')).strip()
    if not segment_signal:
        segment_signal = overall_driver_signal
    signal_focus = _signal_focus(segment_signal)

    if churn_rate > overall_churn_rate and value_is_high:
        key_observation = 'High churn + high LTV'
        business_implication = (
            f'This segment churns at {churn_rate:.1%}, '
            f'which is {churn_gap * 100:.1f} percentage points above '
            'the overall churn rate, '
            f'while still holding above-median lifetime value of {row["Avg_Lifetime_Value"]:,.2f}. '
            f'The strongest model signal is {segment_signal}, indicating the risk is concentrated around {signal_focus}. '
            'This is a retention opportunity with high commercial upside if churn risk is addressed early.'
        )
        recommended_action = (
            f'Prioritize retention outreach for high-value customers, focusing on {signal_focus}, '
            're-engagement, and friction reduction to protect revenue before churn compounds.'
        )
    elif churn_rate > overall_churn_rate:
        key_observation = 'High churn'
        business_implication = (
            f'This segment churns at {churn_rate:.1%}, which is {churn_gap:.1%} points above the overall churn rate. '
            f'The model identifies {segment_signal} as a prominent predictive signal. '
            f'This suggests investigating {signal_focus} as a potential retention lever; '
            'its business impact should be validated through further analysis or controlled testing.'
            'The business should treat this as a targeted intervention segment rather than a broad campaign.'
        )
        recommended_action = (
            f'Run targeted re-engagement and friction-reduction campaigns for this segment, with emphasis on {signal_focus} '
            'and monitoring of at-risk customer behavior.'
        )
    elif value_is_high:
        key_observation = 'High LTV + low churn'
        business_implication = (
            f'This segment keeps churn below the overall baseline at {churn_rate:.1%}, while retaining above-median lifetime value '
            f'of {row["Avg_Lifetime_Value"]:,.2f}. The model signal {segment_signal} suggests the segment remains stable, '
            'but should still be protected through loyalty and proactive service experience management.'
        )
        recommended_action = (
            'Protect loyalty with high-value customer programs, personalized offers, and proactive service experiences '
            "to preserve the segment's strong commercial contribution."
        )
    else:
        key_observation = 'Low churn + lower priority'
        business_implication = (
            f'This segment churns at {churn_rate:.1%}, below the overall benchmark, and contributes with lower value potential. '
            'The current risk posture is manageable, so intervention should stay light and monitor for any shift in engagement.'
        )
        recommended_action = (
            'Maintain the current journey and monitor for early warning signals so the segment does not drift into higher churn risk.'
        )

    recommendation_records.append({
        'Segment': row['Segment'],
        'Segment_Name': row['Segment_Name'],
        'Business_Priority': row['Business_Priority'],
        'Key_Observation': key_observation,
        'Main_Churn_Signal': segment_signal,
        'Business_Implication': business_implication,
        'Recommended_Action': recommended_action,
        'Priority_Rank': row['Retention_Priority_Rank'],
    })

business_recommendations = pd.DataFrame(recommendation_records, columns=[
    'Segment', 'Segment_Name', 'Business_Priority', 'Key_Observation',
    'Main_Churn_Signal', 'Business_Implication', 'Recommended_Action',
    'Priority_Rank',
]).sort_values('Priority_Rank', ascending=True).reset_index(drop=True)
business_recommendations


,Segment,Segment_Name,Business_Priority,Key_Observation,Main_Churn_Signal,Business_Implication,Recommended_Action,Priority_Rank
0,1,At-Risk Customers (Segment 1),HIGH,High churn,num__Lifetime_Value,"This segment churns at 33.9%, which is 5.0% po...",Run targeted re-engagement and friction-reduct...,1
1,0,Loyal High-Value Customers (Segment 0),MAINTAIN,High LTV + low churn,num__Customer_Service_Calls; num__Lifetime_Val...,This segment keeps churn below the overall bas...,Protect loyalty with high-value customer progr...,2


## SECTION 09 - Export and Final Validation

In [ ]:
business_outputs = {
    'overall_churn_summary.csv': overall_churn_summary,
    'segment_business_summary.csv': segment_business_summary,
    'churn_driver_summary.csv': churn_driver_summary,
    'segment_churn_driver_summary.csv': segment_churn_driver_summary,
    'segment_retention_priority.csv': segment_retention_priority,
    'business_recommendations.csv': business_recommendations,
}

for filename, output_frame in business_outputs.items():
    output_frame.to_csv(BUSINESS_DIR / filename, index=False)

if total_customers <= 0:
    raise ValueError('Total Customers must be greater than zero.')
if not set(customer_data[TARGET].unique()).issubset({0, 1}):
    raise ValueError('Churned must contain only 0 and 1.')
if customer_segments['Segment'].nunique() <= 0:
    raise ValueError('At least one customer segment is required.')
if segment_profile['Customer_Count'].sum() != total_customers:
    raise ValueError('Final validation failed: segment customer counts do not match total customers.')
if segment_profile['Churned_Count'].sum() != int(customer_data[TARGET].sum()):
    raise ValueError('Final validation failed: segment churned counts do not match total churned customers.')
if not np.isclose(segment_profile['Customer_Percentage'].sum(), 100.0, atol=0.1):
    raise ValueError('Final validation failed: segment percentages do not sum to approximately 100%.')
calculated_segment_churn = segment_profile['Churned_Count'] / segment_profile['Customer_Count']
if not np.allclose(segment_profile['Churn_Rate'], calculated_segment_churn, atol=1e-6):
    raise ValueError('Final validation failed: segment churn rates do not match churned/customer counts.')
if not segment_profile['Churn_Rate'].between(0, 1).all():
    raise ValueError('Final validation failed: segment churn rates are outside [0, 1].')
if segment_profile['Avg_Lifetime_Value'].dropna().lt(0).any():
    raise ValueError('Final validation failed: average lifetime value is negative.')
rank_values = segment_profile['Retention_Priority_Rank'].dropna().astype(int)
expected_ranks = set(range(1, len(segment_profile) + 1))
actual_ranks = set(rank_values.unique())
if actual_ranks != expected_ranks:
    raise ValueError('Final validation failed: retention priority ranks are not unique and consecutive.')
if segment_profile['Retention_Priority_Rank'].isna().any():
    raise ValueError('Final validation failed: retention priority rank contains null values.')

missing_business_outputs = [
    filename for filename in business_outputs
    if not (BUSINESS_DIR / filename).exists()
]
if missing_business_outputs:
    raise FileNotFoundError(f'Business outputs were not created: {missing_business_outputs}')

print(f'Validated and exported {len(business_outputs)} business CSV files to {BUSINESS_DIR}.')


Validated and exported 6 business CSV files to d:\Desktop\Projects\customer-churn-analytics\reports\business.


: 